In [3]:
import torch
import triton
import triton.language as tl

@triton.jit
def my_kernel(x_ptr, y_ptr, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, x * 2.0, mask=mask)

x = torch.randn(1024 * 1024, device='cuda')
y = torch.empty_like(x)
grid = (triton.cdiv(1024 * 1024, 1024),)

# Warmup (compila y cachea)
for _ in range(10):
    my_kernel[grid](x, y, 1024 * 1024, BLOCK=1024)

# Benchmark real
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(100):
    my_kernel[grid](x, y, 1024 * 1024, BLOCK=1024)
end.record()
torch.cuda.synchronize()
print(f"Latency: {start.elapsed_time(end) / 100:.2f} ms")

/tmp/tmp5uxtsnrr/cuda_utils.c:7:10: fatal error: Python.h: No such file or directory
    7 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.


CalledProcessError: Command '['/usr/bin/gcc', '/tmp/tmp5uxtsnrr/cuda_utils.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmp5uxtsnrr/cuda_utils.cpython-312-x86_64-linux-gnu.so', '-l:libcuda.so.1', '-L/home/uno21/.virtualenvs/traffic-sign-cnn/lib/python3.12/site-packages/triton/backends/nvidia/lib', '-L/lib64', '-I/home/uno21/.virtualenvs/traffic-sign-cnn/lib/python3.12/site-packages/triton/backends/nvidia/include', '-I/tmp/tmp5uxtsnrr', '-I/usr/include/python3.12']' returned non-zero exit status 1.